# Paper results — *Unconditional Flow Matching With Classifier-Based Pruning for Distribution-Aligned Soil Spectral Synthesis*

Regenerates every table and figure in the letter
([10.1109/LGRS.2026.3721837](https://doi.org/10.1109/LGRS.2026.3721837)) from the
committed result files.

| § | Produces | Reads | Runs from a bare clone? |
|---|---|---|---|
| 0 | setup | — | yes |
| 1 | **Table I** — fidelity across variants | `results/flow_matching/*/phase2_test_*.csv` | yes |
| 2 | **Fig. 1** — manifold + spectral | iter_0 artifacts **and** the prepared LUCAS data | **no** |
| 3 | **Fig. 2** — pruning trade-off | `results/flow_matching/*/phase1_*.csv` | yes |
| 4 | **Table II** — method comparison, and runtimes | + `results/tabsyn/*/tabsyn_baseline_*.csv` | yes |
| 5 | four-variant summary (not in the letter) | same as §4 | yes |

Only §2 needs the LUCAS 2015 data, which is distributed by ESDAC under its own
terms and is not redistributed here. It skips itself with an explanatory message
if the data is absent, so *Run all* works either way.

Run §0 first, then any section in any order.

## 0 — Setup

In [ ]:
import os
import sys
import json
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Locate the repository root by walking up to paths.py.
_here = os.path.abspath("")
while _here != os.path.dirname(_here) and not os.path.exists(os.path.join(_here, "paths.py")):
    _here = os.path.dirname(_here)
REPO_ROOT = _here
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from paths import (fm_results, tabsyn_results, prepared_dataset,
                   TABLES_DIR, FIGURES_DIR)

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Variant order and display names; must match Table I of the published letter.
DATASETS = [
    "chem_nospectral",
    "chem_spectral",
    "chem_phys_nospectral",
    "chem_phys_spectral",
]

DATASET_NAMES = {
    "chem_nospectral": "Chemistry Baseline",
    "chem_spectral": "Chemistry + Spectroscopy",
    "chem_phys_nospectral": "Chemistry + Soil Texture",
    "chem_phys_spectral": "Comprehensive Joint Set",
}

# Show result tables in full.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# Shared figure styling
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["legend.fontsize"] = 14

print(f"Repository root: {REPO_ROOT}")
print(f"Tables  -> {os.path.relpath(TABLES_DIR, REPO_ROOT)}")
print(f"Figures -> {os.path.relpath(FIGURES_DIR, REPO_ROOT)}")

## 1 — Table I: generative fidelity across dataset variants

For each variant the optimal pruning threshold is the one minimising the upper
confidence bound (mean + 1 sd) of the **validation** AUC; the reported figures
then come from the held-out **test** partition. Writes
`results/tables/auc_table.tex`.

In [ ]:
results_table = []

for variant in DATASETS:
    results_dir = fm_results(variant)
    if not os.path.exists(results_dir):
        print(f"Warning: Results for {variant} not found at {results_dir}")
        continue
    
    # Load CSVs
    val_results = pd.read_csv(os.path.join(results_dir, "phase1_val_results.csv"))
    test_results = pd.read_csv(os.path.join(results_dir, "phase2_test_results.csv"))
    test_rej = pd.read_csv(os.path.join(results_dir, "phase2_test_rejection_rates.csv"))
    
    # 1. Determine Optimal Threshold on Validation Set
    mean_val = val_results.mean()
    std_val = val_results.std()
    ucb_val = mean_val + std_val
    
    # Find best pruned strategy minimizing UCB
    best_strategy = ucb_val.drop('Pure').idxmin()
    best_threshold = best_strategy.replace('Pruned_', '')
    
    # 2. Extract Definitive Results on Test Set
    pure_test_mean = test_results['Pure'].mean()
    pure_test_std = test_results['Pure'].std()
    
    pruned_test_mean = test_results[best_strategy].mean()
    pruned_test_std = test_results[best_strategy].std()
    
    rej_mean = test_rej[best_threshold].mean() * 100
    rej_std = test_rej[best_threshold].std() * 100
    
    results_table.append({
        "Dataset Variant": DATASET_NAMES[variant],
        "Opt. Threshold": f"{float(best_threshold):.2f}",
        "Pure Baseline (AUC)": f"{pure_test_mean:.4f} $\\pm$ {pure_test_std:.4f}",
        "Optimally Pruned (AUC)": f"{pruned_test_mean:.4f} $\\pm$ {pruned_test_std:.4f}",
        "AUC Improvement": f"{(pure_test_mean - pruned_test_mean):.4f}",
        "Rejection Rate (\\%)": f"{rej_mean:.2f} $\\pm$ {rej_std:.2f}"
    })

df_table = pd.DataFrame(results_table)
display(df_table)

# Export to LaTeX
latex_table = df_table.to_latex(index=False, column_format="lccccc",
                                caption="Generative Fidelity (Test AUC) Across Dataset Variants",
                                label="tab:auc_results")
# Spans both IEEEtran columns, as in the published letter
latex_table = latex_table.replace("\\begin{table}", "\\begin{table*}[!ht]\n\\centering")
latex_table = latex_table.replace("\\end{table}", "\\end{table*}")

out_path = os.path.join(TABLES_DIR, "auc_table.tex")
with open(out_path, "w") as f:
    f.write(latex_table)

print(f"Saved AUC table to {os.path.relpath(out_path, REPO_ROOT)}")

## 2 — Figure 1: manifold alignment and spectral consistency

*Requires the prepared LUCAS data.* UMAP is fitted exclusively on the real test
data, and the pruned synthetic set is projected into that fixed space. The right
panel reconstructs the VNIR–SWIR curves by inverting the spectral PCA.

Uses the iteration-0 artifacts committed under
`results/flow_matching/chem_phys_spectral/saved_samples_iter_0/`, so the figure
can be redrawn without re-running the Monte Carlo loop.

In [ ]:
import joblib
import umap
from sklearn.model_selection import train_test_split

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

umap_plot_mode = "pruned"  # Options: "pure", "pruned", "both"
plots_dir = FIGURES_DIR

for variant in ["chem_phys_spectral"]:
    print(f"\nProcessing {variant}...")
    results_dir = fm_results(variant)
    if not os.path.exists(results_dir):
        continue
        
    val_results = pd.read_csv(os.path.join(results_dir, "phase1_val_results.csv"))
    best_strategy = (val_results.mean() + val_results.std()).drop('Pure').idxmin()
    best_threshold = best_strategy.replace('Pruned_', '')
    
    iter_idx = 0
    iter_dir = os.path.join(results_dir, f"saved_samples_iter_{iter_idx}")
    if not os.path.exists(iter_dir):
        continue
        
    with open(os.path.join(iter_dir, "metadata.json"), 'r') as f:
        metadata = json.load(f)
    seed = metadata['seed']
    
    dataset_csv = prepared_dataset(variant)
    if not os.path.exists(dataset_csv):
        print(
            "\n  [SKIPPED] Figure 1 needs the prepared LUCAS dataset, which is not\n"
            "  redistributed with this repository.\n"
            f"    expected: {os.path.relpath(dataset_csv, REPO_ROOT)}\n"
            "  Obtain LUCAS 2015 from ESDAC and run prepare_datasets.py first;\n"
            "  see the README. Every other section runs without it."
        )
        continue

    df_full = pd.read_csv(dataset_csv)
    
    from flowMatching.data import get_traditional_features
    traditional_features = get_traditional_features(variant)
    
    raw_spectral_features = []
    wavelengths = []
    for col in df_full.columns:
        if col not in traditional_features and col not in ['Point_ID', 'geometry']:
            try:
                wavelengths.append(float(col))
                raw_spectral_features.append(col)
            except ValueError:
                continue
    
    if len(raw_spectral_features) == 0:
        print(f"No spectral features found for {variant}. Skipping Triumphant Visual.")
        continue
        
    sort_idx = np.argsort(wavelengths)
    wavelengths = np.array(wavelengths)[sort_idx]
    raw_spectral_features = np.array(raw_spectral_features)[sort_idx]
    
    if 'cluster' in df_full.columns:
        y_raw = df_full['cluster'].values
    else:
        y_raw = np.zeros(len(df_full))
    class_counts = pd.Series(y_raw).value_counts()
    stratify = y_raw if not (class_counts < 3).any() else None
    
    df_trainval, df_test, _, _ = train_test_split(df_full, y_raw, test_size=0.15, random_state=seed, stratify=stratify)
    
    real_trad = df_test[traditional_features].values
    iter_scalers_dict = joblib.load(os.path.join(iter_dir, 'fm_scaler_iter.pkl'))
    clustering_scaler = iter_scalers_dict['traditional']
    
    real_trad_df = pd.DataFrame(real_trad, columns=traditional_features)
    real_trad_scaled = clustering_scaler.transform(real_trad_df) if hasattr(clustering_scaler, 'feature_names_in_') else clustering_scaler.transform(real_trad)
    
    df_pure_syn = pd.read_parquet(os.path.join(iter_dir, "pure_test.parquet"))
    df_pruned_syn = pd.read_parquet(os.path.join(iter_dir, f"pruned_test_{best_threshold}.parquet"))
    
    pure_trad_scaled = df_pure_syn[traditional_features].values
    pruned_trad_scaled = df_pruned_syn[traditional_features].values
    
    spectral_pca_fitted = iter_scalers_dict['pca']
    spectral_scaler_fitted = iter_scalers_dict['spectral']
    
    real_spectra = df_test[raw_spectral_features].values
    real_spectra_df = pd.DataFrame(real_spectra, columns=raw_spectral_features)
    real_spectra_scaled = spectral_scaler_fitted.transform(real_spectra_df) if hasattr(spectral_scaler_fitted, 'feature_names_in_') else spectral_scaler_fitted.transform(real_spectra)
    
    pca_cols = [c for c in df_pure_syn.columns if c.startswith('Spectral_PC')]
    
    pure_pca_df = pd.DataFrame(df_pure_syn[pca_cols].values, columns=pca_cols)
    pure_spectra_scaled = spectral_pca_fitted.inverse_transform(pure_pca_df) if hasattr(spectral_pca_fitted, 'feature_names_in_') else spectral_pca_fitted.inverse_transform(pure_pca_df.values)
    
    pruned_pca_df = pd.DataFrame(df_pruned_syn[pca_cols].values, columns=pca_cols)
    pruned_spectra_scaled = spectral_pca_fitted.inverse_transform(pruned_pca_df) if hasattr(spectral_pca_fitted, 'feature_names_in_') else spectral_pca_fitted.inverse_transform(pruned_pca_df.values)
    
    X_real = np.hstack([real_trad_scaled, real_spectra_scaled])
    X_pure = np.hstack([pure_trad_scaled, pure_spectra_scaled])
    X_pruned = np.hstack([pruned_trad_scaled, pruned_spectra_scaled])
        
    print("Running UMAP (Fitted rigidly to Real Data)...")
    reducer_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)

    umap_real = reducer_umap.fit_transform(X_real)
    if umap_plot_mode in ["pruned", "both"]:
        umap_pruned = reducer_umap.transform(X_pruned)
    if umap_plot_mode in ["pure", "both"]:
        umap_pure = reducer_umap.transform(X_pure)
    
    fig, ax1 = plt.subplots(figsize=(8, 6))
    
    # Adjusted for grayscale print visibility (darker reference, contrasting synthetic markers/luminance)
    pt_size = 30; real_bg_color = 'dimgray'; real_bg_alpha = 0.8; syn_alpha = 0.7
    ax1.scatter(umap_real[:, 0], umap_real[:, 1], c=real_bg_color, alpha=real_bg_alpha, s=pt_size, marker='o', label='Real (Reference)', zorder=1)
    if umap_plot_mode in ["pure", "both"]:
        ax1.scatter(umap_pure[:, 0], umap_pure[:, 1], c='red', alpha=syn_alpha, s=pt_size, marker='x', label='Pure Synthetic', zorder=2)
    if umap_plot_mode in ["pruned", "both"]:
        ax1.scatter(umap_pruned[:, 0], umap_pruned[:, 1], c='green', alpha=syn_alpha, s=pt_size, marker='^', label=f'Pruned Synthetic (r={best_threshold})', zorder=3)
    
    # ax1.set_title(f"Manifold Alignment (UMAP)\n{DATASET_NAMES[variant]}")
    ax1.set_xlabel("UMAP 1")
    ax1.set_ylabel("UMAP 2")
    
    # Make legend items fully opaque and larger for clarity
    leg = ax1.legend()
    for lh in leg.legend_handles:
        lh.set_alpha(1)
        lh.set_sizes([40])
        
    ax1.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    save_path_umap = os.path.join(plots_dir, f"manifold_alignment_{variant}.png")
    plt.savefig(save_path_umap, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved Manifold Alignment plot to {save_path_umap}")
    
    # Ensure spectral arrays are wrapped in DataFrames for the inverse_transform to prevent UserWarnings
    if hasattr(spectral_scaler_fitted, 'inverse_transform'):
        pure_spectra_scaled_df = pd.DataFrame(pure_spectra_scaled, columns=raw_spectral_features)
        pruned_spectra_scaled_df = pd.DataFrame(pruned_spectra_scaled, columns=raw_spectral_features)
        pure_spectra_raw = spectral_scaler_fitted.inverse_transform(pure_spectra_scaled_df) if hasattr(spectral_scaler_fitted, 'feature_names_in_') else spectral_scaler_fitted.inverse_transform(pure_spectra_scaled)
        pruned_spectra_raw = spectral_scaler_fitted.inverse_transform(pruned_spectra_scaled_df) if hasattr(spectral_scaler_fitted, 'feature_names_in_') else spectral_scaler_fitted.inverse_transform(pruned_spectra_scaled)
    else:
        pure_spectra_raw = pure_spectra_scaled
        pruned_spectra_raw = pruned_spectra_scaled
        
    real_mean = np.mean(real_spectra, axis=0)
    real_std = np.std(real_spectra, axis=0)
    pure_mean = np.mean(pure_spectra_raw, axis=0)
    pure_std = np.std(pure_spectra_raw, axis=0)
    pruned_mean = np.mean(pruned_spectra_raw, axis=0)
    pruned_std = np.std(pruned_spectra_raw, axis=0)
    
    fig, ax2 = plt.subplots(figsize=(8, 6))
    ax2.plot(wavelengths, real_mean, 'k-', label='Real Data', linewidth=2.5)
    ax2.fill_between(wavelengths, real_mean - real_std, real_mean + real_std, color='black', alpha=0.15)
    ax2.plot(wavelengths, pure_mean, 'r--', label='Pure Synthetic', linewidth=2)
    ax2.fill_between(wavelengths, pure_mean - pure_std, pure_mean + pure_std, color='red', alpha=0.1)
    ax2.plot(wavelengths, pruned_mean, 'g-.', label=f'Pruned Synthetic (r={best_threshold})', linewidth=2)
    ax2.fill_between(wavelengths, pruned_mean - pruned_std, pruned_mean + pruned_std, color='green', alpha=0.1)
    
    # ax2.set_title(f"Spectral Reconstruction Consistency\n{DATASET_NAMES[variant]}")
    ax2.set_xlabel("Wavelength (nm)")
    ax2.set_ylabel("Reflectance")
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    save_path_spectra = os.path.join(plots_dir, f"spectral_reconstruction_{variant}.png")
    plt.savefig(save_path_spectra, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved Spectral Reconstruction plot to {save_path_spectra}")

## 3 — Figure 2: pruning trade-off curve

Validation-set dynamics — AUC improvement against sample rejection rate as the
threshold *r* tightens from 1.0 to 0.50. Deliberately plotted on the validation
split rather than the test split, so threshold selection never inspects test data.

In [ ]:
target_variant = "chem_phys_spectral"
results_dir = fm_results(target_variant)
plots_dir = FIGURES_DIR

plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

if os.path.exists(results_dir):
    # Validation-set dynamics; the test split is never used for threshold selection
    val_results = pd.read_csv(os.path.join(results_dir, "phase1_val_results.csv"))
    val_rej = pd.read_csv(os.path.join(results_dir, "phase1_rejection_rates.csv"))
    
    # Get base pure AUC
    mean_pure_auc = val_results['Pure'].mean()
    
    thresholds = []
    auc_improvements = []
    rejection_rates = []
    
    for col in val_rej.columns:
        th = float(col)
        pruned_col = f"Pruned_{col}"
        mean_pruned_auc = val_results[pruned_col].mean()
        
        thresholds.append(th)
        auc_improvements.append(mean_pure_auc - mean_pruned_auc)
        rejection_rates.append(val_rej[col].mean() * 100) # Percentage
    
    # Sort by threshold descending (1.0 to 0.4)
    sorted_indices = np.argsort(thresholds)[::-1]
    thresholds = np.array(thresholds)[sorted_indices]
    auc_improvements = np.array(auc_improvements)[sorted_indices]
    rejection_rates = np.array(rejection_rates)[sorted_indices]
    
    fig, ax1 = plt.subplots(figsize=(8, 5))

    color = 'tab:blue'
    ax1.set_xlabel('Pruning Threshold ($r$)', fontsize=12)
    ax1.set_ylabel('Validation AUC Improvement (Absolute Drop)', color=color, fontsize=12)
    
    # Solid line with circles
    lns1 = ax1.plot(thresholds, auc_improvements, color=color, marker='o', 
                    linestyle='-', linewidth=2, label='AUC Improvement')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.invert_xaxis()

    ax2 = ax1.twinx()  
    color = 'tab:red'
    
    ax2.set_ylabel('Sample Rejection Rate (%)', color=color, fontsize=12)
    # Dashed line with squares
    lns2 = ax2.plot(thresholds, rejection_rates, color=color, marker='s', 
                    linestyle='--', linewidth=2, label='Rejection Rate')
    ax2.tick_params(axis='y', labelcolor=color)

    # --- Combine Legends into one ---
    lns = lns1 + lns2
    labs = [l.get_label() for l in lns]
    # Placing the legend at a visible spot, e.g., upper left or center left
    ax1.legend(lns, labs, loc='upper left', frameon=True)

    # --- Tick and Axis Tweaks ---
    desired_xticks = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
    ax1.set_xticks(desired_xticks)
    ax1.set_xticklabels([f'{t:.1f}' for t in desired_xticks])
    
    from matplotlib.ticker import MultipleLocator
    ax1.yaxis.set_major_locator(MultipleLocator(0.01))   
    min_rej = rejection_rates.min()
    max_rej = rejection_rates.max()
    pad = (max_rej - min_rej) * 0.1 if max_rej != min_rej else 1
    ax2.set_ylim(min_rej - pad, max_rej + pad)          
    ax2.yaxis.set_major_locator(MultipleLocator(5))     

    fig.tight_layout()
    # plt.title(f"Pruning Trade-off ({dataset_names[target_variant]})")
    
    save_path = os.path.join(plots_dir, "tradeoff_curve.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved Trade-off Curve to {save_path}")
else:
    print(f"Results for {target_variant} not found. Cannot plot trade-off curve.")

## 4 — Table II: comparison against TabSyn, and runtimes

Our method at 100 Monte Carlo iterations against the TabSyn baseline at 20.
Iteration *i* uses `seed = 42 + i` in both pipelines, so the two methods see
identical splits and the AUCs are directly comparable.

The runtimes printed at the end are the figures quoted in the letter. Both are
**inclusive**: ours brackets Flow Matching training plus the pruning-tree fit,
validation generation and one XGBoost AUC; TabSyn's brackets its dataset
preprocessing plus VAE plus diffusion training. The per-sample rates in the text
divide the sampling times by the number of rows produced — 5.684 s ÷ 2,979 for
TabSyn, 0.1017 s ÷ 639 for ours.

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np

# Paths for the Comprehensive Joint Set
variant = "chem_phys_spectral"
gst_results_dir = fm_results(variant)
tabsyn_results_dir = tabsyn_results(variant)

# ---------------------------------------------------------
# 1. Process Pure and Pruned Data (Generative Stress Testing)
# ---------------------------------------------------------
val_results = pd.read_csv(os.path.join(gst_results_dir, "phase1_val_results.csv"))
test_results = pd.read_csv(os.path.join(gst_results_dir, "phase2_test_results.csv"))
test_results_10x = pd.read_csv(os.path.join(gst_results_dir, "phase2_test_results_10x.csv"))
test_times = pd.read_csv(os.path.join(gst_results_dir, "phase2_test_generation_times.csv"))

# Determine Optimal Threshold on Validation Set
mean_val = val_results.mean()
std_val = val_results.std()
best_strategy = (mean_val + std_val).drop('Pure').idxmin()
best_threshold = best_strategy.replace('Pruned_', '')

# Extract 1x AUC
pure_1x_mean, pure_1x_std = test_results['Pure'].mean(), test_results['Pure'].std()
pruned_1x_mean, pruned_1x_std = test_results[best_strategy].mean(), test_results[best_strategy].std()

# Extract 10x AUC
pure_10x_mean, pure_10x_std = test_results_10x['Pure'].mean(), test_results_10x['Pure'].std()
pruned_10x_mean, pruned_10x_std = test_results_10x[best_strategy].mean(), test_results_10x[best_strategy].std()

# ---------------------------------------------------------
# 2. Process TabSyn Data
# ---------------------------------------------------------
tabsyn_files = glob.glob(os.path.join(tabsyn_results_dir, "tabsyn_baseline_*.csv"))

if not tabsyn_files:
    print(f"Warning: No TabSyn files found in {tabsyn_results_dir}")
    df_tabsyn = pd.DataFrame()
else:
    df_tabsyn = pd.concat([pd.read_csv(f) for f in tabsyn_files], ignore_index=True)

# Calculate 1x and 10x TabSyn AUC
ts_1x_mean, ts_1x_std = df_tabsyn['TabSyn_Test_AUC'].mean(), df_tabsyn['TabSyn_Test_AUC'].std()
ts_10x_mean, ts_10x_std = df_tabsyn['TabSyn_Test_AUC_10x'].mean(), df_tabsyn['TabSyn_Test_AUC_10x'].std()

# ---------------------------------------------------------
# 3. Build the Compact Results Table
# ---------------------------------------------------------
# Added \textbf{} to highlight the Pruned method's results
comparison_table = [
    {
        "Method": "Pure",
        "AUC (1x)": f"{pure_1x_mean:.4f} $\\pm$ {pure_1x_std:.4f}",
        "AUC (10x)": f"{pure_10x_mean:.4f} $\\pm$ {pure_10x_std:.4f}"
    },
    {
        "Method": f"Pruned ({float(best_threshold):.2f})",
        "AUC (1x)": f"\\textbf{{{pruned_1x_mean:.4f} $\\pm$ {pruned_1x_std:.4f}}}",
        "AUC (10x)": f"\\textbf{{{pruned_10x_mean:.4f} $\\pm$ {pruned_10x_std:.4f}}}"
    },
    {
        "Method": "TabSyn",
        "AUC (1x)": f"{ts_1x_mean:.4f} $\\pm$ {ts_1x_std:.4f}",
        "AUC (10x)": f"{ts_10x_mean:.4f} $\\pm$ {ts_10x_std:.4f}"
    }
]

df_comparison = pd.DataFrame(comparison_table)
display(df_comparison)

# ---------------------------------------------------------
# 4. Export to Compact LaTeX
# ---------------------------------------------------------
# Added escape=False so pandas doesn't break the LaTeX formatting commands
latex_table = df_comparison.to_latex(index=False, escape=False, column_format="lcc",
                                     caption="Method Comparison on Comprehensive Joint Set",
                                     label="tab:chem_phys_spectral_scaling")

# Single-column table, as in the published letter.
latex_table = latex_table.replace("\\begin{table}", "\\begin{table}[!ht]\n\\centering")
latex_table = latex_table.replace(
    "\\begin{tabular}", "\\footnotesize\n\\setlength{\\tabcolsep}{3pt}\n\\begin{tabular}")
latex_table = latex_table.replace("\\end{tabular}", "\\end{tabular}%")

out_path = os.path.join(TABLES_DIR, "chem_phys_spectral_comparison.tex")
with open(out_path, "w") as f:
    f.write(latex_table)

print(f"\nSaved comparison table to {os.path.relpath(out_path, REPO_ROOT)}")

# ---------------------------------------------------------
# 5. Extract Runtimes for Manuscript Text
# ---------------------------------------------------------
# Per-iteration timings, with a fallback to the raw metadata.json files.
train_times_csv = os.path.join(gst_results_dir, "train_times.csv")
if os.path.exists(train_times_csv):
    train_times = pd.read_csv(train_times_csv)["time_train_sec"].tolist()
else:
    train_times = [
        json.load(open(m)).get("time_train_sec", 0)
        for m in glob.glob(os.path.join(gst_results_dir, "saved_samples_iter_*", "metadata.json"))
    ]

print("\n" + "-"*50)
print("RUNTIME METRICS (For your manuscript discussion):")
print("-"*50)
print(f"Flow Matching Train Time (Mean): {np.mean(train_times):.2f}s")
print(f"Pure Sampling Time (Mean):       {test_times['Pure'].mean():.2f}s")
print(f"Pruned Sampling Time (Mean):     {test_times[best_strategy].mean():.2f}s")
print(f"TabSyn Train Time (Mean):        {df_tabsyn['Time_Train_Sec'].mean():.2f}s")
print(f"TabSyn 1x Sample Time (Mean):    {df_tabsyn['Time_Sample_Sec'].mean():.2f}s")
print(f"TabSyn 10x Sample Time (Mean):   {df_tabsyn['Time_Sample_10x_Sec'].mean():.2f}s")
print("-"*50)

## 5 — Four-variant summary

The same head-to-head comparison across all four variants. Only the Comprehensive
Joint Set rows appear as Table II in the letter; the remaining variants are quoted
inline in the Benchmark Comparison subsection.

In [ ]:
import os
import glob
import pandas as pd

summary_list = []

for variant in DATASETS:
    gst_results_dir = fm_results(variant)
    tabsyn_results_dir = tabsyn_results(variant)
    
    # ---------------------------------------------------------
    # 1. Process Pure and Pruned Data
    # ---------------------------------------------------------
    try:
        val_results = pd.read_csv(os.path.join(gst_results_dir, "phase1_val_results.csv"))
        test_results = pd.read_csv(os.path.join(gst_results_dir, "phase2_test_results.csv"))
        test_results_10x = pd.read_csv(os.path.join(gst_results_dir, "phase2_test_results_10x.csv"))
        
        # Determine Optimal Threshold
        mean_val = val_results.mean()
        std_val = val_results.std()
        best_strategy = (mean_val + std_val).drop('Pure').idxmin()
        best_threshold = float(best_strategy.replace('Pruned_', ''))
        
        # Extract 1x AUC
        pure_1x_mean, pure_1x_std = test_results['Pure'].mean(), test_results['Pure'].std()
        pruned_1x_mean, pruned_1x_std = test_results[best_strategy].mean(), test_results[best_strategy].std()
        
        # Extract 10x AUC
        pure_10x_mean, pure_10x_std = test_results_10x['Pure'].mean(), test_results_10x['Pure'].std()
        pruned_10x_mean, pruned_10x_std = test_results_10x[best_strategy].mean(), test_results_10x[best_strategy].std()
    
    except FileNotFoundError:
        print(f"Warning: Missing GST files for {variant}")
        continue

    # ---------------------------------------------------------
    # 2. Process TabSyn Data
    # ---------------------------------------------------------
    tabsyn_files = glob.glob(os.path.join(tabsyn_results_dir, "tabsyn_baseline_*.csv"))
    
    if not tabsyn_files:
        print(f"Warning: No TabSyn files found for {variant}")
        ts_1x_mean, ts_1x_std = 0.0, 0.0
        ts_10x_mean, ts_10x_std = 0.0, 0.0
    else:
        df_tabsyn = pd.concat([pd.read_csv(f) for f in tabsyn_files], ignore_index=True)
        ts_1x_mean, ts_1x_std = df_tabsyn['TabSyn_Test_AUC'].mean(), df_tabsyn['TabSyn_Test_AUC'].std()
        ts_10x_mean, ts_10x_std = df_tabsyn['TabSyn_Test_AUC_10x'].mean(), df_tabsyn['TabSyn_Test_AUC_10x'].std()
        
    # ---------------------------------------------------------
    # 3. Append to Summary Table
    # ---------------------------------------------------------
    summary_list.extend([
        {
            "Dataset Variant": DATASET_NAMES[variant],
            "Method": "Pure",
            "AUC (1x)": f"{pure_1x_mean:.4f} $\\pm$ {pure_1x_std:.4f}",
            "AUC (10x)": f"{pure_10x_mean:.4f} $\\pm$ {pure_10x_std:.4f}"
        },
        {
            "Dataset Variant": DATASET_NAMES[variant],
            "Method": f"Pruned ({best_threshold:.2f})",
            "AUC (1x)": f"\\textbf{{{pruned_1x_mean:.4f} $\\pm$ {pruned_1x_std:.4f}}}",
            "AUC (10x)": f"\\textbf{{{pruned_10x_mean:.4f} $\\pm$ {pruned_10x_std:.4f}}}"
        },
        {
            "Dataset Variant": DATASET_NAMES[variant],
            "Method": "TabSyn",
            "AUC (1x)": f"{ts_1x_mean:.4f} $\\pm$ {ts_1x_std:.4f}",
            "AUC (10x)": f"{ts_10x_mean:.4f} $\\pm$ {ts_10x_std:.4f}"
        }
    ])

df_summary = pd.DataFrame(summary_list)
display(df_summary)